# LLM 02: Transformer Architecture

Build the core pieces of a decoder-only Transformer in PyTorch:
1. Scaled dot-product attention (+ causal mask)
2. Multi-head attention
3. A full Transformer block (MHA + FFN + residuals + LayerNorm)
4. Inspect attention patterns
5. Exercises: RoPE, KV caching

Deps: `pip install torch matplotlib`

## 1. Scaled Dot-Product Attention with Causal Mask

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

seq_len, d_k = 5, 8
Q = torch.randn(seq_len, d_k)
K = torch.randn(seq_len, d_k)
V = torch.randn(seq_len, d_k)

causal = torch.tril(torch.ones(seq_len, seq_len))
out, w = scaled_dot_product_attention(Q, K, V, mask=causal)
print('attention weights (lower-triangular because of causal mask):')
print(w.round(decimals=2))
print('\noutput shape:', out.shape)

## 2. Multi-Head Attention Module

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, T, C = x.shape
        Q = self.W_q(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        scores = Q @ K.transpose(-2, -1) / (self.d_head ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        out = weights @ V
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.W_o(out), weights

mha = MultiHeadAttention(d_model=64, n_heads=4)
x = torch.randn(2, 10, 64)   # batch=2, seq=10, dim=64
mask = torch.tril(torch.ones(10, 10))
out, w = mha(x, mask)
print('output shape:', out.shape)
print('attention weights shape:', w.shape, '(B, heads, T, T)')

## 3. A Full Transformer Block

`Block(x) = x + MHA(LN(x)); out = x + FFN(LN(x))` — pre-norm style, matches Llama/GPT-3+.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.0):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, mask=None):
        attn_out, _ = self.attn(self.ln1(x), mask)
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x

block = TransformerBlock(d_model=64, n_heads=4)
x = torch.randn(2, 10, 64)
mask = torch.tril(torch.ones(10, 10))
print('block output shape:', block(x, mask).shape)
print('param count:', sum(p.numel() for p in block.parameters()))

## 4. Visualize Attention Patterns

Inspect which tokens attend to which after a forward pass. Real LLMs show interpretable patterns (diagonal, previous-token, first-token "sink", etc.) after training.

In [ ]:
import matplotlib.pyplot as plt

_, w = mha(torch.randn(1, 8, 64), torch.tril(torch.ones(8, 8)))
w = w.squeeze(0).detach()   # (heads, T, T)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, ax in enumerate(axes):
    ax.imshow(w[i], vmin=0, vmax=1)
    ax.set_title(f'head {i}')
    ax.set_xlabel('key position')
    ax.set_ylabel('query position')
plt.tight_layout()
plt.show()

## 5. Exercise: Add RoPE (Rotary Position Embedding)

The block above has no position information — the model is permutation-invariant. Modern LLMs use RoPE: rotate Q and K by an angle that depends on position. Sketch:

```python
def apply_rope(x, pos):
    # x: (..., d_head)  assume d_head even
    d = x.shape[-1]
    theta = 10000 ** (-torch.arange(0, d, 2) / d)
    angles = pos[:, None] * theta[None, :]
    cos, sin = torch.cos(angles), torch.sin(angles)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin,
                        x1 * sin + x2 * cos], dim=-1).flatten(-2)
```

**Try it:** extend `MultiHeadAttention` to apply RoPE to Q and K before the dot product. Verify that shuffling tokens now changes the output.

## 6. Exercise: KV Caching for Fast Generation

During autoregressive generation, K and V for previously-generated tokens don't change. Production inference stacks (vLLM, TGI) cache them.

**Try it:** write a `generate(model, prompt_ids, max_new=20)` function that
1. Runs the full prefill once.
2. Keeps `K, V` tensors alongside the hidden state.
3. On each step, only computes Q for the new token and appends K, V to the cache.

Measure the speedup vs re-running the whole sequence each step. Expect ~`seq_len`× faster as `seq_len` grows.

## 7. Takeaways

- **Attention = content-based softmax-weighted average of Values.**
- **Multi-head = parallel subspace specializations.**
- **Transformer block = attention + FFN + residuals + LayerNorm.**
- **RoPE is the current-best position encoding** for long-context decoder-only models.
- **KV caching is the #1 inference optimization** for generation.

Next: **03 — Tokenization**, where we peel back the layer below embeddings and see how text becomes integers.